In [ ]:
import pandas as pd
import numpy as np
import time
import statsmodels.api as sm
from tqdm import tqdm
import os
from google.colab import drive
drive.mount('/content/drive')
# set directory to your data files
path_data = '/content/drive/My Drive/invs'
os.listdir(path_data)


# Step 1: Preparing the CRSP data
import pandas as pd
import numpy as np
import time

print("Preparing CRSP data...")
start_time = time.time()

# Load CRSP data
crsp = pd.read_csv(path_data + '/market.csv')

# Convert all column names to lowercase
crsp.columns = crsp.columns.str.lower()

# Convert the 'date' column to datetime format (your format is 'yyyy/mm/dd')
crsp['date'] = pd.to_datetime(crsp['date'], format='%Y-%m-%d')
# Convert return values to numeric (if not already)
crsp['ret'] = pd.to_numeric(crsp['ret'], errors='coerce')

# Compute market capitalization (in millions): |price| × shares outstanding
crsp['mktcap'] = crsp['prc'].abs() * crsp['shrout']

# Extract year and month from date
crsp['year'] = crsp['date'].dt.year
crsp['month'] = crsp['date'].dt.month


# Keep only common stocks (share codes 10 and 11)
crsp = crsp[crsp['shrcd'].isin([10, 11])]

# Keep only stocks listed on NYSE, AMEX, or NASDAQ (exchange codes 1, 2, 3)
crsp = crsp[crsp['exchcd'].isin([1, 2, 3])]

# Remove any duplicates (one observation per stock per month)
crsp = crsp.drop_duplicates(subset=['date', 'permno'])
crsp = crsp[~crsp['mktcap'].isna()]

# -------------------------------
# Construct Book Equity (Hou-Xue-Zhang method)
# -------------------------------
book = pd.read_csv(path_data + '/book.csv')
book.columns = book.columns.str.strip().str.lower()

# ---------------------------------------------
# Book Equity construction (Hou, Xue, Zhang 2020 full fallback version)
# ---------------------------------------------

# Construct Book Equity (BE) using fallback logic: seq → ceq + pstk → at - lt
book['be'] = book['seq'].fillna(book['ceq'] + book['pstk']).fillna(book['at'] - book['lt'])

# Adjust BE by adding deferred taxes and subtracting preferred stock
book['be'] = book['be'] + book['txditc'].fillna(0) - book['pstk'].fillna(0)

# Set BE to NaN if invalid
book.loc[book['be'] <= 0, 'be'] = np.nan

# Drop rows with missing required fields
book = book.dropna(subset=['fyear', 'be', 'lpermno'])


print(f"[INFO] Book Equity constructed using full HXZ fallback method.")


# Step 3: Sort stocks into portfolios based on Book-to-Market ratio (Fama-French style)
print("Creating portfolios based on Book-to-Market ratio (Fama-French method)...")
start_time = time.time()

# 1. Prepare Compustat (Book) data
book['fyear'] = book['fyear'].astype(int)
book['target_year'] = book['fyear'] + 1  # Align with June of year t


# 2. Use only June market cap from CRSP (filtered for common stocks and valid exchanges)
crsp_june = crsp[
    (crsp['month'] == 6) &
    (crsp['shrcd'].isin([10, 11])) &
    (crsp['exchcd'].isin([1, 2, 3]))
].copy()

crsp_june = crsp_june[['permno', 'year', 'mktcap']]

crsp_june = crsp_june.dropna(subset=['mktcap'])

# 3. Merge Book and Market Cap using INNER JOIN (only keep companies with both)
bm_data = pd.merge(
    crsp_june,
    book[['lpermno', 'target_year', 'be']],
    left_on=['permno', 'year'],
    right_on=['lpermno', 'target_year'],
    how='inner'
)

# 4. Compute Book-to-Market ratio
bm_data['bm_ratio'] = bm_data['be'] / bm_data['mktcap']


# Report matching quality
print(f"Matching companies with both Book & June Market Cap: {len(bm_data):,}")

# 5. Prepare to build portfolios
portfolios = []
nportfolios = 5

exclude_years = [1987] + list(range(1997, 1999)) + list(range(2000, 2003)) + list(range(2007, 2010))

for year in tqdm(range(1982, 2019), desc="Processing years"):
    if year in exclude_years:
        continue


    # Get B/M data from June of year t (used to sort)
    sorting_data = bm_data[bm_data['year'] == year][['permno', 'bm_ratio']].copy()

    if sorting_data.empty:
        print(f"No valid B/M data for {year}")
        continue

    # Assign quantile rank (0 = low B/M, 4 = high B/M)
    sorting_data['quantile_rank'] = pd.qcut(
        sorting_data['bm_ratio'],
        nportfolios,
        labels=False,
        duplicates='drop'
    )

    # Get return data from July of year t to June of year t+1
    returns_window = crsp[
        ((crsp['year'] == year) & (crsp['month'] >= 7)) |
        ((crsp['year'] == year + 1) & (crsp['month'] <= 6))
    ]

    annual_portfolios = []

    for q in range(nportfolios):
        stocks = sorting_data[sorting_data['quantile_rank'] == q]['permno'].tolist()
        if not stocks:
            print(f"Empty portfolio Q{q} in {year}")
            continue

        quantile_returns = returns_window[returns_window['permno'].isin(stocks)]
        if quantile_returns.empty:
            continue

        return_matrix = quantile_returns.pivot_table(
            index='date',
            columns='permno',
            values='ret',
            aggfunc='mean'
        ).iloc[1:]  # drop first row for alignment

        portfolio_returns = return_matrix.mean(axis=1)
        portfolio_returns.name = str(q)
        annual_portfolios.append(portfolio_returns)

    if annual_portfolios:
        portfolios.append(pd.concat(annual_portfolios, axis=1))

# Combine all portfolio returns across years
final_portfolios = pd.concat(portfolios, axis=0) if portfolios else pd.DataFrame()


# Step 4a: Merge Portfolio Returns with Fama-French Factors
# Load Fama-French 3-factor monthly data
ff = pd.read_csv(path_data + '/F-F_Research_Data_Factors.CSV')

# Drop non-data rows (e.g., footnotes)
ff = ff[pd.to_numeric(ff.iloc[:, 0], errors='coerce').notnull()].copy()
ff.rename(columns={ff.columns[0]: 'date'}, inplace=True)

# Convert date to integer and extract year/month
ff['date'] = ff['date'].astype(int)
ff['year'] = ff['date'] // 100
ff['month'] = ff['date'] % 100

# Rename factor columns
ff.rename(columns={
    'Mkt-RF': 'ExMkt',
    'SMB': 'SMB',
    'HML': 'HML',
    'RF': 'RF'
}, inplace=True)

# Convert percentages to decimals
factor_cols = ['ExMkt', 'SMB', 'HML', 'RF']
ff[factor_cols] = ff[factor_cols].astype(float) / 100

# Ensure 'portfolios' is a DataFrame and index is datetime
if isinstance(portfolios, list):
    portfolios = pd.concat(portfolios, axis=0)

# Prepare portfolio returns for merging
portfolios_ff = portfolios.copy()
portfolios_ff['year'] = portfolios_ff.index.to_series().dt.year
portfolios_ff['month'] = portfolios_ff.index.to_series().dt.month

# Merge on year and month
portfolios_ff = pd.merge(portfolios_ff, ff, on=['year', 'month'], how='inner')

# Calculate excess returns for each portfolio
for col in portfolios.columns:
    portfolios_ff[f'{col}_excess'] = portfolios_ff[col] - portfolios_ff['RF']


# Step 4b: Regressions
import statsmodels.api as sm

print("\nRunning CAPM and Fama-French 3-factor regressions...\n")

# Ensure portfolio is a DataFrame (in case it was created as list)
if isinstance(portfolios, list):
    portfolios = pd.concat(portfolios, axis=0)

# Compute average annualized portfolio returns
avg_returns = ((1 + portfolios.mean(axis=0)) ** 12 - 1) * 100
print("Average Portfolio Returns (Annualized, %):")
print(avg_returns.round(4), "\n")

# Recalculate excess returns just in case
for p in range(nportfolios):
    portfolios_ff[f'ExRet_{p}'] = portfolios_ff[str(p)].astype(float) - portfolios_ff['RF'].astype(float)

# CAPM Regression
table_capm = []

for p in range(nportfolios):
    y = portfolios_ff[f'ExRet_{p}']
    X = sm.add_constant(portfolios_ff['ExMkt'])
    results = sm.OLS(y, X).fit()

    row = pd.DataFrame({
    'alpha': results.params['const'],
    'beta_mkt': results.params['ExMkt'],
    'alpha_t': results.tvalues['const'],
    'alpha_pval': results.pvalues['const'],
    'beta_mkt_t': results.tvalues['ExMkt'],
    'beta_mkt_p': results.pvalues['ExMkt'],
    'rmse': np.sqrt(results.mse_resid),
    'R2': results.rsquared
    }, index=[p])

    table_capm.append(row)

table_capm = pd.concat(table_capm)
table_capm.index.name = 'quintile'
table_capm['alpha_annual'] = table_capm['alpha'] * 12
table_capm['significant_alpha'] = table_capm['alpha_pval'] < 0.05

print("CAPM Regression Results:")
print(table_capm.round(5), "\n")

# Fama-French 3-Factor Regression
table_ff = []

for p in range(nportfolios):
    y = portfolios_ff[f'ExRet_{p}']
    X = sm.add_constant(portfolios_ff[['ExMkt', 'SMB', 'HML']])
    results = sm.OLS(y, X).fit()

    row = pd.DataFrame({
    'alpha': results.params['const'],
    'beta_mkt': results.params['ExMkt'],
    'beta_smb': results.params['SMB'],
    'beta_hml': results.params['HML'],
    'alpha_t': results.tvalues['const'],
    'alpha_pval': results.pvalues['const'],
    'beta_mkt_t': results.tvalues['ExMkt'],
    'beta_mkt_p': results.pvalues['ExMkt'],
    'beta_smb_t': results.tvalues['SMB'],
    'beta_smb_p': results.pvalues['SMB'],
    'beta_hml_t': results.tvalues['HML'],
    'beta_hml_p': results.pvalues['HML'],
    'rmse': np.sqrt(results.mse_resid),
    'R2': results.rsquared
    }, index=[p])

    table_ff.append(row)

table_ff = pd.concat(table_ff)
table_ff.index.name = 'quintile'
table_ff['alpha_annual'] = table_ff['alpha'] * 12
table_ff['significant_alpha'] = table_ff['alpha_pval'] < 0.05

print("Fama-French 3-Factor Regression Results:")
print(table_ff.round(5))
vw_portfolios = final_portfolios.copy()
vw_portfolios_ff = portfolios_ff.copy()
vw_table_capm = table_capm.copy()
vw_table_ff = table_ff.copy()



Mounted at /content/drive
Preparing CRSP data...
[INFO] Book Equity constructed using full HXZ fallback method.
Creating portfolios based on Book-to-Market ratio (Fama-French method)...


<ipython-input-1-9c7dff4255de>:82: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  book['fyear'] = book['fyear'].astype(int)
<ipython-input-1-9c7dff4255de>:83: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  book['target_year'] = book['fyear'] + 1  # Align with June of year t


Matching companies with both Book & June Market Cap: 186,406


Processing years:   5%|▌         | 2/37 [00:00<00:02, 16.13it/s]

No valid B/M data for 1982


Processing years: 100%|██████████| 37/37 [00:02<00:00, 14.64it/s]



Running CAPM and Fama-French 3-factor regressions...

Average Portfolio Returns (Annualized, %):
0    12.2299
1    15.6713
2    17.3031
3    18.5971
4    22.1602
dtype: float64 

CAPM Regression Results:
            alpha  beta_mkt  alpha_t  alpha_pval  beta_mkt_t  beta_mkt_p  \
quintile                                                                   
0        -0.00587   1.23621 -2.94380     0.00350    22.84468         0.0   
1        -0.00234   1.13829 -1.65614     0.09876    29.73399         0.0   
2         0.00008   1.01590  0.06131     0.95115    27.85266         0.0   
3         0.00202   0.91569  1.55001     0.12221    25.87114         0.0   
4         0.00490   0.87901  2.46653     0.01421    16.31271         0.0   

             rmse       R2  alpha_annual  significant_alpha  
quintile                                                     
0         0.03304  0.63887      -0.07043               True  
1         0.02338  0.74981      -0.02803              False  
2         0.02

In [ ]:
import pandas as pd
import numpy as np
import time
import statsmodels.api as sm
from tqdm import tqdm
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')
path_data = '/content/drive/My Drive/invs'
os.listdir(path_data)

# ------------------------------
# Step 1: Load & Clean CRSP Data
# ------------------------------
print("Preparing CRSP data...")
crsp = pd.read_csv(path_data + '/market.csv')
crsp.columns = crsp.columns.str.lower()
crsp['date'] = pd.to_datetime(crsp['date'], format='%Y-%m-%d')
crsp['ret'] = pd.to_numeric(crsp['ret'], errors='coerce')
crsp['mktcap'] = crsp['prc'].abs() * crsp['shrout']
crsp['year'] = crsp['date'].dt.year
crsp['month'] = crsp['date'].dt.month
crsp = crsp[crsp['shrcd'].isin([10, 11])]
crsp = crsp[crsp['exchcd'].isin([1, 2, 3])]
crsp = crsp.drop_duplicates(subset=['date', 'permno'])
crsp = crsp[~crsp['mktcap'].isna()]

# ------------------------------
# Step 2: Book Equity (HXZ Fallback)
# ------------------------------
book = pd.read_csv(path_data + '/book.csv')
book.columns = book.columns.str.strip().str.lower()
# Construct Book Equity (BE) using fallback logic: seq → ceq + pstk → at - lt
book['be'] = book['seq'].fillna(book['ceq'] + book['pstk']).fillna(book['at'] - book['lt'])

# Adjust BE by adding deferred taxes and subtracting preferred stock
book['be'] = book['be'] + book['txditc'].fillna(0) - book['pstk'].fillna(0)

# Set BE to NaN if invalid
book.loc[book['be'] <= 0, 'be'] = np.nan

# Drop rows with missing required fields
book = book.dropna(subset=['fyear', 'be', 'lpermno'])

print("[INFO] Book Equity constructed using full HXZ fallback method.")

# ------------------------------
# Step 3: Portfolio Construction (Value-Weighted)
# ------------------------------
book['fyear'] = book['fyear'].astype(int)
book['target_year'] = book['fyear'] + 1
crsp_june = crsp[(crsp['month'] == 6)].copy()
crsp_june = crsp_june[['permno', 'year', 'mktcap']].dropna()
bm_data = pd.merge(
    crsp_june,
    book[['lpermno', 'target_year', 'be']],
    left_on=['permno', 'year'],
    right_on=['lpermno', 'target_year'],
    how='inner'
)

bm_data['bm_ratio'] = bm_data['be'] / bm_data['mktcap']
print(f"[INFO] Matching: {len(bm_data):,}")

nportfolios = 5
portfolios = []

exclude_years = [1987] + list(range(1997, 1999)) + list(range(2000, 2003)) + list(range(2007, 2010))


for year in tqdm(range(1982, 2019), desc="Processing years"):
    if year in exclude_years:
        continue
    sorting_data = bm_data[bm_data['year'] == year][['permno', 'bm_ratio']].copy()
    if sorting_data.empty:
        continue
    sorting_data['quantile_rank'] = pd.qcut(sorting_data['bm_ratio'], nportfolios, labels=False, duplicates='drop')
    returns_window = crsp[((crsp['year'] == year) & (crsp['month'] >= 7)) |
                          ((crsp['year'] == year + 1) & (crsp['month'] <= 6))]
    annual_portfolios = []

    for q in range(nportfolios):
        stocks = sorting_data[sorting_data['quantile_rank'] == q]['permno'].tolist()
        if not stocks:
            continue
        quantile_returns = returns_window[returns_window['permno'].isin(stocks)]
        if quantile_returns.empty:
            continue
        return_matrix = quantile_returns.pivot_table(index='date', columns='permno', values='ret', aggfunc='mean')
        mktcap_matrix = quantile_returns.pivot_table(index='date', columns='permno', values='mktcap', aggfunc='mean')
        vwret = (return_matrix * mktcap_matrix).sum(axis=1) / mktcap_matrix.sum(axis=1)
        vwret.name = str(q)
        annual_portfolios.append(vwret)

    if annual_portfolios:
        combined = pd.concat(annual_portfolios, axis=1)
        combined.columns = [str(i) for i in range(combined.shape[1])]
        portfolios.append(combined)

final_portfolios = pd.concat(portfolios, axis=0) if portfolios else pd.DataFrame()

# ------------------------------
# Step 4: Merge with FF Factors
# ------------------------------
ff = pd.read_csv(path_data + '/F-F_Research_Data_Factors.CSV')
ff = ff[pd.to_numeric(ff.iloc[:, 0], errors='coerce').notnull()].copy()
ff.rename(columns={ff.columns[0]: 'date'}, inplace=True)
ff['date'] = ff['date'].astype(int)
ff['year'] = ff['date'] // 100
ff['month'] = ff['date'] % 100
ff.rename(columns={'Mkt-RF': 'ExMkt', 'SMB': 'SMB', 'HML': 'HML', 'RF': 'RF'}, inplace=True)
ff[['ExMkt', 'SMB', 'HML', 'RF']] = ff[['ExMkt', 'SMB', 'HML', 'RF']].astype(float) / 100

portfolios_ff = final_portfolios.copy()
portfolios_ff['year'] = portfolios_ff.index.to_series().dt.year
portfolios_ff['month'] = portfolios_ff.index.to_series().dt.month
portfolios_ff = pd.merge(portfolios_ff, ff, on=['year', 'month'], how='inner')

for col in final_portfolios.columns:
    portfolios_ff[f'{col}_excess'] = portfolios_ff[col] - portfolios_ff['RF']

# ------------------------------
# Step 5: Regressions
# ------------------------------
print("\nRunning CAPM and Fama-French 3-factor regressions...\n")
avg_returns = ((1 + final_portfolios.mean(axis=0)) ** 12 - 1) * 100
print("Average Portfolio Returns (Annualized, %):")
print(avg_returns.round(4), "\n")

table_capm = []
table_ff = []

for p in range(nportfolios):
    y = portfolios_ff[f'{p}_excess']
    # CAPM
    X_capm = sm.add_constant(portfolios_ff['ExMkt'])
    results_capm = sm.OLS(y, X_capm).fit()
    table_capm.append({
    'alpha': results_capm.params['const'],
    'beta_mkt': results_capm.params['ExMkt'],
    'alpha_t': results_capm.tvalues['const'],
    'alpha_pval': results_capm.pvalues['const'],
    'beta_mkt_t': results_capm.tvalues['ExMkt'],
    'beta_mkt_p': results_capm.pvalues['ExMkt'],
    'rmse': np.sqrt(results_capm.mse_resid),
    'R2': results_capm.rsquared
    })


    # FF 3-factor
    X_ff = sm.add_constant(portfolios_ff[['ExMkt', 'SMB', 'HML']])
    results_ff = sm.OLS(y, X_ff).fit()
    table_ff.append({
    'alpha': results_ff.params['const'],
    'beta_mkt': results_ff.params['ExMkt'],
    'beta_smb': results_ff.params['SMB'],
    'beta_hml': results_ff.params['HML'],
    'alpha_t': results_ff.tvalues['const'],
    'alpha_pval': results_ff.pvalues['const'],
    'beta_mkt_t': results_ff.tvalues['ExMkt'],
    'beta_mkt_p': results_ff.pvalues['ExMkt'],
    'beta_smb_t': results_ff.tvalues['SMB'],
    'beta_smb_p': results_ff.pvalues['SMB'],
    'beta_hml_t': results_ff.tvalues['HML'],
    'beta_hml_p': results_ff.pvalues['HML'],
    'rmse': np.sqrt(results_ff.mse_resid),
    'R2': results_ff.rsquared
      })


# Format tables
table_capm = pd.DataFrame(table_capm, index=[f'Q{i}' for i in range(nportfolios)])
table_capm['alpha_annual'] = table_capm['alpha'] * 12
table_capm['significant_alpha'] = table_capm['alpha_pval'] < 0.05

table_ff = pd.DataFrame(table_ff, index=[f'Q{i}' for i in range(nportfolios)])
table_ff['alpha_annual'] = table_ff['alpha'] * 12
table_ff['significant_alpha'] = table_ff['alpha_pval'] < 0.05

print("CAPM Results:")
print(table_capm.round(5), "\n")

print("Fama-French 3-Factor Results:")
print(table_ff.round(5))
eq_portfolios = portfolios.copy()
eq_portfolios_ff = portfolios_ff.copy()
eq_table_capm = table_capm.copy()
eq_table_ff = table_ff.copy()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Preparing CRSP data...
[INFO] Book Equity constructed using full HXZ fallback method.
[INFO] Matching: 186,406


Processing years: 100%|██████████| 37/37 [00:03<00:00, 10.31it/s]


Running CAPM and Fama-French 3-factor regressions...

Average Portfolio Returns (Annualized, %):
0    25.2059
1    22.7405
2    22.5124
3    21.4559
4    25.8003
dtype: float64 

CAPM Results:
      alpha  beta_mkt  alpha_t  alpha_pval  beta_mkt_t  beta_mkt_p     rmse  \
Q0  0.00489   1.12649  5.15465         0.0    43.88131         0.0  0.01646   
Q1  0.00441   1.00174  7.14721         0.0    59.88870         0.0  0.01072   
Q2  0.00486   0.93934  7.09652         0.0    50.59802         0.0  0.01190   
Q3  0.00449   0.90227  4.65410         0.0    34.51854         0.0  0.01676   
Q4  0.00744   0.90568  5.88948         0.0    26.47256         0.0  0.02193   

         R2  alpha_annual  significant_alpha  
Q0  0.85673       0.05863               True  
Q1  0.91762       0.05297               True  
Q2  0.88828       0.05837               True  
Q3  0.78725       0.05390               True  
Q4  0.68518       0.08927               True   

Fama-French 3-Factor Results:
      alpha  beta

In [ ]:
# Define a function that builds the summary table
def build_summary_table(portfolios, portfolios_ff, table_capm, table_ff, nportfolios=5):
    from scipy.stats import ttest_1samp, skew, kurtosis
    import statsmodels.api as sm
    import pandas as pd

    summary_table = pd.DataFrame(columns=[f'Qnt {i+1}' for i in range(nportfolios)] + ['Long-Short'], index=[
        'Mean', 't-stat', 'p-value', 'SD', 'Skewness', 'Kurtosis'
    ])
    portfolios['Long-Short'] = portfolios[str(nportfolios - 1)] - portfolios['0']

    for i in range(nportfolios):
        data = portfolios[str(i)].dropna()
        t_stat, p_val = ttest_1samp(data, 0)
        summary_table.loc['Mean', f'Qnt {i+1}'] = data.mean()
        summary_table.loc['t-stat', f'Qnt {i+1}'] = t_stat
        summary_table.loc['p-value', f'Qnt {i+1}'] = p_val
        summary_table.loc['SD', f'Qnt {i+1}'] = data.std()
        summary_table.loc['Skewness', f'Qnt {i+1}'] = skew(data)
        summary_table.loc['Kurtosis', f'Qnt {i+1}'] = kurtosis(data)

    longshort = portfolios['Long-Short'].dropna()
    t_stat, p_val = ttest_1samp(longshort, 0)
    summary_table.loc['Mean', 'Long-Short'] = longshort.mean()
    summary_table.loc['t-stat', 'Long-Short'] = t_stat
    summary_table.loc['p-value', 'Long-Short'] = p_val
    summary_table.loc['SD', 'Long-Short'] = longshort.std()
    summary_table.loc['Skewness', 'Long-Short'] = skew(longshort)
    summary_table.loc['Kurtosis', 'Long-Short'] = kurtosis(longshort)

    # The table for CAPM
    capm_table_fmt = pd.DataFrame(columns=[f'Qnt {i+1}' for i in range(nportfolios)] + ['Long-Short'], index=[
        'Alpha', 'Alpha t', 'Alpha p',
        'Beta Mkt', 'Beta Mkt t', 'Beta Mkt p',
        'R2'
    ])
    y_ls = portfolios_ff[str(nportfolios - 1)] - portfolios_ff['0'] - portfolios_ff['RF']
    X_capm = sm.add_constant(portfolios_ff['ExMkt'])
    result_ls = sm.OLS(y_ls, X_capm).fit()
    capm_table_fmt.loc['Alpha', 'Long-Short'] = result_ls.params['const']
    capm_table_fmt.loc['Alpha t', 'Long-Short'] = result_ls.tvalues['const']
    capm_table_fmt.loc['Alpha p', 'Long-Short'] = result_ls.pvalues['const']
    capm_table_fmt.loc['Beta Mkt', 'Long-Short'] = result_ls.params['ExMkt']
    capm_table_fmt.loc['Beta Mkt t', 'Long-Short'] = result_ls.tvalues['ExMkt']
    capm_table_fmt.loc['Beta Mkt p', 'Long-Short'] = result_ls.pvalues['ExMkt']
    capm_table_fmt.loc['R2', 'Long-Short'] = result_ls.rsquared

    for i in range(nportfolios):
        row = table_capm.iloc[i]
        capm_table_fmt.loc['Alpha', f'Qnt {i+1}'] = row['alpha']
        capm_table_fmt.loc['Alpha t', f'Qnt {i+1}'] = row['alpha_t']
        capm_table_fmt.loc['Alpha p', f'Qnt {i+1}'] = row['alpha_pval']
        capm_table_fmt.loc['Beta Mkt', f'Qnt {i+1}'] = row['beta_mkt']
        capm_table_fmt.loc['Beta Mkt t', f'Qnt {i+1}'] = row['beta_mkt_t']
        capm_table_fmt.loc['Beta Mkt p', f'Qnt {i+1}'] = row['beta_mkt_p']
        capm_table_fmt.loc['R2', f'Qnt {i+1}'] = row['R2']

    # The table for FF 3-factor
    ff_table_fmt = pd.DataFrame(columns=[f'Qnt {i+1}' for i in range(nportfolios)] + ['Long-Short'], index=[
        'Alpha', 'Alpha t', 'Alpha p',
        'Beta Mkt', 'Beta Mkt t', 'Beta Mkt p',
        'Beta SMB', 'Beta SMB t', 'Beta SMB p',
        'Beta HML', 'Beta HML t', 'Beta HML p',
        'R2'
    ])
    X_ff = sm.add_constant(portfolios_ff[['ExMkt', 'SMB', 'HML']])
    result_ff_ls = sm.OLS(y_ls, X_ff).fit()
    ff_table_fmt.loc['Alpha', 'Long-Short'] = result_ff_ls.params['const']
    ff_table_fmt.loc['Alpha t', 'Long-Short'] = result_ff_ls.tvalues['const']
    ff_table_fmt.loc['Alpha p', 'Long-Short'] = result_ff_ls.pvalues['const']
    ff_table_fmt.loc['Beta Mkt', 'Long-Short'] = result_ff_ls.params['ExMkt']
    ff_table_fmt.loc['Beta Mkt t', 'Long-Short'] = result_ff_ls.tvalues['ExMkt']
    ff_table_fmt.loc['Beta Mkt p', 'Long-Short'] = result_ff_ls.pvalues['ExMkt']
    ff_table_fmt.loc['Beta SMB', 'Long-Short'] = result_ff_ls.params['SMB']
    ff_table_fmt.loc['Beta SMB t', 'Long-Short'] = result_ff_ls.tvalues['SMB']
    ff_table_fmt.loc['Beta SMB p', 'Long-Short'] = result_ff_ls.pvalues['SMB']
    ff_table_fmt.loc['Beta HML', 'Long-Short'] = result_ff_ls.params['HML']
    ff_table_fmt.loc['Beta HML t', 'Long-Short'] = result_ff_ls.tvalues['HML']
    ff_table_fmt.loc['Beta HML p', 'Long-Short'] = result_ff_ls.pvalues['HML']
    ff_table_fmt.loc['R2', 'Long-Short'] = result_ff_ls.rsquared

    for i in range(nportfolios):
        row = table_ff.iloc[i]
        ff_table_fmt.loc['Alpha', f'Qnt {i+1}'] = row['alpha']
        ff_table_fmt.loc['Alpha t', f'Qnt {i+1}'] = row['alpha_t']
        ff_table_fmt.loc['Alpha p', f'Qnt {i+1}'] = row['alpha_pval']
        ff_table_fmt.loc['Beta Mkt', f'Qnt {i+1}'] = row['beta_mkt']
        ff_table_fmt.loc['Beta Mkt t', f'Qnt {i+1}'] = row['beta_mkt_t']
        ff_table_fmt.loc['Beta Mkt p', f'Qnt {i+1}'] = row['beta_mkt_p']
        ff_table_fmt.loc['Beta SMB', f'Qnt {i+1}'] = row['beta_smb']
        ff_table_fmt.loc['Beta SMB t', f'Qnt {i+1}'] = row['beta_smb_t']
        ff_table_fmt.loc['Beta SMB p', f'Qnt {i+1}'] = row['beta_smb_p']
        ff_table_fmt.loc['Beta HML', f'Qnt {i+1}'] = row['beta_hml']
        ff_table_fmt.loc['Beta HML t', f'Qnt {i+1}'] = row['beta_hml_t']
        ff_table_fmt.loc['Beta HML p', f'Qnt {i+1}'] = row['beta_hml_p']
        ff_table_fmt.loc['R2', f'Qnt {i+1}'] = row['R2']

    return summary_table.round(4), capm_table_fmt.round(4), ff_table_fmt.round(4)
    # Ensure portfolios is a DataFrame, not list
if isinstance(eq_portfolios, list):
    eq_portfolios = pd.concat(eq_portfolios, axis=0)

if isinstance(vw_portfolios, list):
    vw_portfolios = pd.concat(vw_portfolios, axis=0)

# Example usage (equal-weighted or value-weighted)
summary_eq, capm_eq, ff_eq = build_summary_table(eq_portfolios, eq_portfolios_ff, eq_table_capm, eq_table_ff)
summary_vw, capm_vw, ff_vw = build_summary_table(vw_portfolios, vw_portfolios_ff, vw_table_capm, vw_table_ff)

print("EQUAL-WEIGHTED SUMMARY:")
print(summary_eq)
print("\nEQUAL-WEIGHTED CAPM:")
print(capm_eq)
print("\nEQUAL-WEIGHTED FF3:")
print(ff_eq)

print("\n\nVALUE-WEIGHTED SUMMARY:")
print(summary_vw)
print("\nVALUE-WEIGHTED CAPM:")
print(capm_vw)
print("\nVALUE-WEIGHTED FF3:")
print(ff_vw)


EQUAL-WEIGHTED SUMMARY:
             Qnt 1     Qnt 2     Qnt 3     Qnt 4     Qnt 5 Long-Short
Mean      0.018909  0.017222  0.017064   0.01633  0.019311   0.000402
t-stat    7.847049  8.327219  8.647225  8.112714  8.921784   0.210046
p-value        0.0       0.0       0.0       0.0       0.0   0.833764
SD        0.043375  0.037226  0.035521  0.036233  0.038961   0.034468
Skewness  0.127295  0.085863  0.042551 -0.071621  0.002408    0.51939
Kurtosis  0.685151  0.882168  0.702045  1.058057  0.929761   1.829796

EQUAL-WEIGHTED CAPM:
                Qnt 1      Qnt 2     Qnt 3      Qnt 4      Qnt 5 Long-Short
Alpha        0.004886   0.004414  0.004864   0.004491   0.007439  -0.000537
Alpha t       5.15465   7.147215   7.09652   4.654099   5.889483  -0.276028
Alpha p           0.0        0.0       0.0   0.000005        0.0   0.782704
Beta Mkt     1.126486   1.001737  0.939336   0.902267   0.905676  -0.216439
Beta Mkt t  43.881314  59.888699  50.59802  34.518541  26.472559  -4.108962
Beta Mkt

In [ ]:
# ---------------------------
# VALUE-WEIGHTED: Pre vs Post 2000 Analysis
# ---------------------------

# Initialize the dataset for pre and post year 2000
vw_ff_pre2000 = vw_portfolios_ff[vw_portfolios_ff['year'] < 2000].copy()
vw_ff_post2000 = vw_portfolios_ff[vw_portfolios_ff['year'] >= 2000].copy()

# Function to run the regression on capm and ff3
def run_regression_split(data, nportfolios=5):
    capm_results = []
    ff3_results = []
    for p in range(nportfolios):
        y = data[f'{p}'] - data['RF']

        # CAPM
        X_capm = sm.add_constant(data['ExMkt'])
        capm_model = sm.OLS(y, X_capm).fit()
        capm_results.append({
            'alpha': capm_model.params['const'],
            'alpha_t': capm_model.tvalues['const'],
            'alpha_p': capm_model.pvalues['const'],
            'R2': capm_model.rsquared
        })

        # FF3
        X_ff = sm.add_constant(data[['ExMkt', 'SMB', 'HML']])
        ff_model = sm.OLS(y, X_ff).fit()
        ff3_results.append({
            'alpha': ff_model.params['const'],
            'alpha_t': ff_model.tvalues['const'],
            'alpha_p': ff_model.pvalues['const'],
            'R2': ff_model.rsquared
        })

    return pd.DataFrame(capm_results, index=[f'Q{i}' for i in range(nportfolios)]), \
           pd.DataFrame(ff3_results, index=[f'Q{i}' for i in range(nportfolios)])


# Run regression
capm_pre, ff3_pre = run_regression_split(vw_ff_pre2000)
capm_post, ff3_post = run_regression_split(vw_ff_post2000)

# Annual alpha calculations
capm_pre['alpha_annual'] = capm_pre['alpha'] * 12
capm_post['alpha_annual'] = capm_post['alpha'] * 12
ff3_pre['alpha_annual'] = ff3_pre['alpha'] * 12
ff3_post['alpha_annual'] = ff3_post['alpha'] * 12

print("\nCAPM (Pre-2000):")
print(capm_pre.round(4))

print("\nCAPM (Post-2000):")
print(capm_post.round(4))

print("\nFF3 (Pre-2000):")
print(ff3_pre.round(4))

print("\nFF3 (Post-2000):")
print(ff3_post.round(4))


# ---------------------------
# Optional: Long-Short Analysis
# ---------------------------
def long_short_alpha(data, label):
    long_short = data['4'] - data['0']
    y = long_short - data['RF']
    X_capm = sm.add_constant(data['ExMkt'])
    X_ff3 = sm.add_constant(data[['ExMkt', 'SMB', 'HML']])
    capm = sm.OLS(y, X_capm).fit()
    ff3 = sm.OLS(y, X_ff3).fit()
    print(f"\n{label} Long-Short CAPM alpha: {round(capm.params['const']*12,4)} (t={round(capm.tvalues['const'],2)})")
    print(f"{label} Long-Short FF3 alpha:  {round(ff3.params['const']*12,4)} (t={round(ff3.tvalues['const'],2)})")

long_short_alpha(vw_ff_pre2000, "Pre-2000")
long_short_alpha(vw_ff_post2000, "Post-2000")


CAPM (Pre-2000):
     alpha  alpha_t  alpha_p      R2  alpha_annual
Q0 -0.0099  -3.2018   0.0017  0.5905       -0.1188
Q1 -0.0039  -1.7176   0.0880  0.6875       -0.0465
Q2 -0.0002  -0.0960   0.9237  0.6552       -0.0024
Q3  0.0016   0.8799   0.3803  0.6591        0.0194
Q4  0.0056   1.8826   0.0618  0.3629        0.0674

CAPM (Post-2000):
     alpha  alpha_t  alpha_p      R2  alpha_annual
Q0 -0.0019  -0.7901   0.4307  0.7068       -0.0234
Q1 -0.0009  -0.5441   0.5872  0.8262       -0.0108
Q2  0.0002   0.1490   0.8818  0.8062        0.0029
Q3  0.0023   1.2900   0.1991  0.7427        0.0277
Q4  0.0040   1.5854   0.1150  0.6056        0.0481

FF3 (Pre-2000):
     alpha  alpha_t  alpha_p      R2  alpha_annual
Q0 -0.0067  -3.9957   0.0001  0.8854       -0.0807
Q1 -0.0024  -2.0607   0.0411  0.9239       -0.0283
Q2 -0.0002  -0.1882   0.8510  0.8915       -0.0027
Q3  0.0006   0.5192   0.6044  0.8838        0.0069
Q4  0.0036   1.8961   0.0600  0.7615        0.0428

FF3 (Post-2000):
     alpha

In [ ]:
# ---------------------------
# EQUAL-WEIGHTED: Pre vs Post 2000 分析
# ---------------------------

# Initialize the dataset for pre and post year 2000
eq_ff_pre2000 = eq_portfolios_ff[eq_portfolios_ff['year'] < 2000].copy()
eq_ff_post2000 = eq_portfolios_ff[eq_portfolios_ff['year'] >= 2000].copy()

# Initialize the regression function
def run_regression_split(data, nportfolios=5):
    capm_results = []
    ff3_results = []
    for p in range(nportfolios):
        y = data[f'{p}'] - data['RF']

        # CAPM
        X_capm = sm.add_constant(data['ExMkt'])
        capm_model = sm.OLS(y, X_capm).fit()
        capm_results.append({
            'alpha': capm_model.params['const'],
            'alpha_t': capm_model.tvalues['const'],
            'alpha_p': capm_model.pvalues['const'],
            'R2': capm_model.rsquared
        })

        # FF3
        X_ff = sm.add_constant(data[['ExMkt', 'SMB', 'HML']])
        ff_model = sm.OLS(y, X_ff).fit()
        ff3_results.append({
            'alpha': ff_model.params['const'],
            'alpha_t': ff_model.tvalues['const'],
            'alpha_p': ff_model.pvalues['const'],
            'R2': ff_model.rsquared
        })

    return pd.DataFrame(capm_results, index=[f'Q{i}' for i in range(nportfolios)]), \
           pd.DataFrame(ff3_results, index=[f'Q{i}' for i in range(nportfolios)])


# Run regression
eq_capm_pre, eq_ff3_pre = run_regression_split(eq_ff_pre2000)
eq_capm_post, eq_ff3_post = run_regression_split(eq_ff_post2000)

# Annual alpha calculations
eq_capm_pre['alpha_annual'] = eq_capm_pre['alpha'] * 12
eq_capm_post['alpha_annual'] = eq_capm_post['alpha'] * 12
eq_ff3_pre['alpha_annual'] = eq_ff3_pre['alpha'] * 12
eq_ff3_post['alpha_annual'] = eq_ff3_post['alpha'] * 12

print("\n[Equal-Weighted] CAPM (Pre-2000):")
print(eq_capm_pre.round(4))

print("\n[Equal-Weighted] CAPM (Post-2000):")
print(eq_capm_post.round(4))

print("\n[Equal-Weighted] FF3 (Pre-2000):")
print(eq_ff3_pre.round(4))

print("\n[Equal-Weighted] FF3 (Post-2000):")
print(eq_ff3_post.round(4))


# ---------------------------
# Optional: Long-Short Alpha (Equal-Weighted)
# ---------------------------
def long_short_alpha(data, label):
    long_short = data['4'] - data['0']
    y = long_short - data['RF']
    X_capm = sm.add_constant(data['ExMkt'])
    X_ff3 = sm.add_constant(data[['ExMkt', 'SMB', 'HML']])
    capm = sm.OLS(y, X_capm).fit()
    ff3 = sm.OLS(y, X_ff3).fit()
    print(f"\n{label} Long-Short CAPM alpha: {round(capm.params['const']*12,4)} (t={round(capm.tvalues['const'],2)})")
    print(f"{label} Long-Short FF3 alpha:  {round(ff3.params['const']*12,4)} (t={round(ff3.tvalues['const'],2)})")

long_short_alpha(eq_ff_pre2000, "Equal-Weighted Pre-2000")
long_short_alpha(eq_ff_post2000, "Equal-Weighted Post-2000")



[Equal-Weighted] CAPM (Pre-2000):
     alpha  alpha_t  alpha_p      R2  alpha_annual
Q0  0.0045   2.8833   0.0045  0.8520        0.0537
Q1  0.0034   4.4845   0.0000  0.9497        0.0404
Q2  0.0045   4.9799   0.0000  0.9035        0.0542
Q3  0.0043   3.8572   0.0002  0.8502        0.0511
Q4  0.0082   4.6375   0.0000  0.6613        0.0982

[Equal-Weighted] CAPM (Post-2000):
     alpha  alpha_t  alpha_p      R2  alpha_annual
Q0  0.0056   5.7224   0.0000  0.8910        0.0676
Q1  0.0057   6.1757   0.0000  0.8838        0.0686
Q2  0.0051   4.9947   0.0000  0.8737        0.0617
Q3  0.0046   2.9053   0.0042  0.7339        0.0554
Q4  0.0064   3.6049   0.0004  0.7204        0.0768

[Equal-Weighted] FF3 (Pre-2000):
     alpha  alpha_t  alpha_p      R2  alpha_annual
Q0  0.0081   8.1058   0.0000  0.9426        0.0973
Q1  0.0035   4.5818   0.0000  0.9503        0.0425
Q2  0.0035   4.1323   0.0001  0.9210        0.0422
Q3  0.0017   2.3067   0.0224  0.9387        0.0203
Q4  0.0046   3.7876   0.0002

# Five factor

In [ ]:
# ------------------------------
# FF5: Equal-Weighted Regression
# ------------------------------
# Step 1: Load FF5 dataset
ff5 = pd.read_csv(path_data + '/F-F_Research_Data_5_Factors_2x3.csv')
ff5 = ff5[pd.to_numeric(ff5.iloc[:, 0], errors='coerce').notnull()].copy()
ff5.rename(columns={ff5.columns[0]: 'date'}, inplace=True)
ff5['date'] = ff5['date'].astype(int)
ff5['year'] = ff5['date'] // 100
ff5['month'] = ff5['date'] % 100
ff5.rename(columns={
    'Mkt-RF': 'ExMkt',
    'SMB': 'SMB',
    'HML': 'HML',
    'RMW': 'RMW',
    'CMA': 'CMA',
    'RF': 'RF'
}, inplace=True)
ff5[['ExMkt', 'SMB', 'HML', 'RMW', 'CMA', 'RF']] = ff5[['ExMkt', 'SMB', 'HML', 'RMW', 'CMA', 'RF']].astype(float) / 100

# Step 2: Merge FF5 factors
eq_portfolios_ff5 = eq_portfolios.copy()
eq_portfolios_ff5['year'] = eq_portfolios_ff5.index.to_series().dt.year
eq_portfolios_ff5['month'] = eq_portfolios_ff5.index.to_series().dt.month
eq_portfolios_ff5 = pd.merge(eq_portfolios_ff5, ff5, on=['year', 'month'], how='inner')

# Step 3: Calculate the excess
for col in eq_portfolios.columns:
    eq_portfolios_ff5[f'{col}_excess'] = eq_portfolios_ff5[col] - eq_portfolios_ff5['RF']

# Step 4: Regression
print("\nEqual-Weighted FF5 Regression results: \n")
eq_ff5_table = []

for p in range(nportfolios):
    y = eq_portfolios_ff5[f'{p}_excess']
    X = sm.add_constant(eq_portfolios_ff5[['ExMkt', 'SMB', 'HML', 'RMW', 'CMA']])
    results = sm.OLS(y, X).fit()
    eq_ff5_table.append({
        'alpha': results.params['const'],
        'alpha_t': results.tvalues['const'],
        'alpha_p': results.pvalues['const'],
        'beta_mkt': results.params['ExMkt'],
        'beta_smb': results.params['SMB'],
        'beta_hml': results.params['HML'],
        'beta_rmw': results.params['RMW'],
        'beta_cma': results.params['CMA'],
        'R2': results.rsquared
    })

eq_ff5_table = pd.DataFrame(eq_ff5_table, index=[f'Q{i}' for i in range(nportfolios)])
eq_ff5_table['alpha_annual'] = eq_ff5_table['alpha'] * 12
eq_ff5_table['significant_alpha'] = eq_ff5_table['alpha_p'] < 0.05

print(eq_ff5_table.round(5))



Equal-Weighted FF5 Regression results: 

      alpha  alpha_t  alpha_p  beta_mkt  beta_smb  beta_hml  beta_rmw  \
Q0  0.00618  9.30832  0.00000   1.03692  -0.02405  -0.50178   0.13032   
Q1  0.00346  5.94738  0.00000   1.03164  -0.00819  -0.08000   0.23145   
Q2  0.00363  5.93682  0.00000   1.00463  -0.04048   0.16658   0.07400   
Q3  0.00223  3.61971  0.00034   1.01968   0.04110   0.49659   0.13174   
Q4  0.00577  6.85369  0.00000   1.00544   0.20832   0.72985  -0.07952   

    beta_cma       R2  alpha_annual  significant_alpha  
Q0  -0.04856  0.93545       0.07421               True  
Q1   0.13328  0.93296       0.04152               True  
Q2   0.11708  0.91860       0.04353               True  
Q3   0.08931  0.92061       0.02674               True  
Q4  -0.01156  0.87171       0.06925               True  


In [ ]:
# ------------------------------
# FF5: Value-Weighted Regression
# ------------------------------
# Step 1: Merge FF5 factors
vw_portfolios_ff5 = vw_portfolios.copy()
vw_portfolios_ff5['year'] = vw_portfolios_ff5.index.to_series().dt.year
vw_portfolios_ff5['month'] = vw_portfolios_ff5.index.to_series().dt.month
vw_portfolios_ff5 = pd.merge(vw_portfolios_ff5, ff5, on=['year', 'month'], how='inner')

# Step 2: Calculate excess
for col in vw_portfolios.columns:
    vw_portfolios_ff5[f'{col}_excess'] = vw_portfolios_ff5[col] - vw_portfolios_ff5['RF']

# Step 3: Regression
print("\n Value-Weighted FF5 Regression results: \n")
vw_ff5_table = []

for p in range(nportfolios):
    y = vw_portfolios_ff5[f'{p}_excess']
    X = sm.add_constant(vw_portfolios_ff5[['ExMkt', 'SMB', 'HML', 'RMW', 'CMA']])
    results = sm.OLS(y, X).fit()
    vw_ff5_table.append({
        'alpha': results.params['const'],
        'alpha_t': results.tvalues['const'],
        'alpha_p': results.pvalues['const'],
        'beta_mkt': results.params['ExMkt'],
        'beta_smb': results.params['SMB'],
        'beta_hml': results.params['HML'],
        'beta_rmw': results.params['RMW'],
        'beta_cma': results.params['CMA'],
        'R2': results.rsquared
    })

vw_ff5_table = pd.DataFrame(vw_ff5_table, index=[f'Q{i}' for i in range(nportfolios)])
vw_ff5_table['alpha_annual'] = vw_ff5_table['alpha'] * 12
vw_ff5_table['significant_alpha'] = vw_ff5_table['alpha_p'] < 0.05

print(vw_ff5_table.round(5))



 Value-Weighted FF5 Regression results: 

      alpha  alpha_t  alpha_p  beta_mkt  beta_smb  beta_hml  beta_rmw  \
Q0 -0.00317 -2.82684  0.00503   1.03857   0.78776  -0.25347  -0.24700   
Q1 -0.00176 -2.17614  0.03035   1.03786   0.76314   0.06155   0.08686   
Q2 -0.00024 -0.32138  0.74815   0.96501   0.76508   0.30346   0.12642   
Q3  0.00110  1.56404  0.11890   0.90773   0.70105   0.35617   0.08082   
Q4  0.00405  3.01096  0.00283   0.86944   0.85527   0.51059  -0.07955   

    beta_cma       R2  alpha_annual  significant_alpha  
Q0  -0.07699  0.89522      -0.03801               True  
Q1  -0.02305  0.92411      -0.02117               True  
Q2   0.00407  0.92112      -0.00289              False  
Q3   0.16977  0.91873       0.01316              False  
Q4   0.22511  0.77875       0.04855               True  


Double Sorting on B/M × Size

In [ ]:

# Equal-Weighted 2x5 Sorting (Size × B/M) + CAPM/FF3/FF5
# -------------------------
double_portfolios_eq = []
n_bm = 5
# Exclude the years where financial crises took place
exclude_years = [1987] + list(range(1997, 1999)) + list(range(2000, 2003)) + list(range(2007, 2010))
for year in tqdm(range(1982, 2019), desc="EQ 2x5"):
    if year in exclude_years: continue
    temp = bm_data[bm_data['year'] == year].copy()
    if temp.empty: continue

    size_median = temp['mktcap'].median()
    temp['size_group'] = np.where(temp['mktcap'] <= size_median, 'S', 'B')

    year_returns = crsp[((crsp['year'] == year) & (crsp['month'] >= 7)) |
                        ((crsp['year'] == year + 1) & (crsp['month'] <= 6))]

    year_result = []
    for size in ['S', 'B']:
        group = temp[temp['size_group'] == size].copy()
        group['bm_rank'] = pd.qcut(group['bm_ratio'], n_bm, labels=False, duplicates='drop')

        for b in range(n_bm):
            stock_ids = group[group['bm_rank'] == b]['permno']
            subset = year_returns[year_returns['permno'].isin(stock_ids)]
            if subset.empty: continue

            rmat = subset.pivot_table(index='date', columns='permno', values='ret')
            ew_ret = rmat.mean(axis=1)
            ew_ret.name = f'{size}{b}'
            year_result.append(ew_ret)

    if year_result:
        double_portfolios_eq.append(pd.concat(year_result, axis=1))

eq_double = pd.concat(double_portfolios_eq, axis=0)
eq_double['year'] = eq_double.index.to_series().dt.year
eq_double['month'] = eq_double.index.to_series().dt.month
eq_double_ff = pd.merge(eq_double, ff5, on=['year', 'month'], how='inner')

EQ 2x5: 100%|██████████| 37/37 [00:04<00:00,  8.38it/s]


In [ ]:
eq_results = []

for col in eq_double.columns:
    if col in ['year', 'month']: continue
    y = eq_double_ff[col] - eq_double_ff['RF']

    capm_X = sm.add_constant(eq_double_ff[['ExMkt']])
    ff3_X = sm.add_constant(eq_double_ff[['ExMkt', 'SMB', 'HML']])
    ff5_X = sm.add_constant(eq_double_ff[['ExMkt', 'SMB', 'HML', 'RMW', 'CMA']])

    capm = sm.OLS(y, capm_X).fit()
    ff3 = sm.OLS(y, ff3_X).fit()
    ff5_model = sm.OLS(y, ff5_X).fit()

    eq_results.append({
        'portfolio': col,
        'capm_alpha': capm.params['const'], 'capm_t': capm.tvalues['const'],
        'ff3_alpha': ff3.params['const'], 'ff3_t': ff3.tvalues['const'],
        'ff5_alpha': ff5_model.params['const'], 'ff5_t': ff5_model.tvalues['const'],
        'ff5_alpha_annual': ff5_model.params['const'] * 12,
        'R2_ff5': ff5_model.rsquared
    })

eq_double_table = pd.DataFrame(eq_results)
print("\nEQ 2×5 Double Sort Results (CAPM / FF3 / FF5):")
print(eq_double_table.round(4))



EQ 2×5 Double Sort Results (CAPM / FF3 / FF5):
  portfolio  capm_alpha  capm_t  ff3_alpha   ff3_t  ff5_alpha   ff5_t  \
0        S0     -0.0086 -3.0061    -0.0057 -2.8943    -0.0046 -2.4061   
1        S1     -0.0029 -1.3314    -0.0012 -0.8402    -0.0010 -0.7182   
2        S2      0.0024  1.2550     0.0033  2.5521     0.0030  2.2535   
3        S3      0.0035  1.9725     0.0040  3.3197     0.0038  3.0885   
4        S4      0.0066  2.6808     0.0067  3.7448     0.0064  3.4902   
5        B0     -0.0053 -3.2467    -0.0022 -2.9946    -0.0014 -1.8927   
6        B1     -0.0023 -2.2783    -0.0011 -1.7845    -0.0015 -2.3859   
7        B2     -0.0009 -0.9660    -0.0008 -1.2832    -0.0016 -2.7251   
8        B3     -0.0005 -0.4912    -0.0011 -1.9600    -0.0017 -3.2169   
9        B4      0.0001  0.1138    -0.0010 -1.6197    -0.0019 -3.0930   

   ff5_alpha_annual  R2_ff5  
0           -0.0548  0.7660  
1           -0.0126  0.7938  
2            0.0359  0.7735  
3            0.0459  0.7642 

In [ ]:
# -------------------------
# Value-Weighted 2x5 Sorting (Size × B/M) + CAPM/FF3/FF5
# -------------------------
double_portfolios_vw = []
n_bm = 5
exclude_years = [1987] + list(range(1997, 1999)) + list(range(2000, 2003)) + list(range(2007, 2010))
for year in tqdm(range(1982, 2019), desc="VW 2x5"):
    if year in exclude_years: continue
    temp = bm_data[bm_data['year'] == year].copy()
    if temp.empty: continue

    size_median = temp['mktcap'].median()
    temp['size_group'] = np.where(temp['mktcap'] <= size_median, 'S', 'B')

    year_returns = crsp[((crsp['year'] == year) & (crsp['month'] >= 7)) |
                        ((crsp['year'] == year + 1) & (crsp['month'] <= 6))]

    year_result = []
    for size in ['S', 'B']:
        group = temp[temp['size_group'] == size].copy()
        group['bm_rank'] = pd.qcut(group['bm_ratio'], n_bm, labels=False, duplicates='drop')

        for b in range(n_bm):
            stock_ids = group[group['bm_rank'] == b]['permno']
            subset = year_returns[year_returns['permno'].isin(stock_ids)]
            if subset.empty: continue

            rmat = subset.pivot_table(index='date', columns='permno', values='ret')
            smat = subset.pivot_table(index='date', columns='permno', values='mktcap')
            vw_ret = (rmat * smat).sum(axis=1) / smat.sum(axis=1)
            vw_ret.name = f'{size}{b}'
            year_result.append(vw_ret)

    if year_result:
        double_portfolios_vw.append(pd.concat(year_result, axis=1))

vw_double = pd.concat(double_portfolios_vw, axis=0)
vw_double['year'] = vw_double.index.to_series().dt.year
vw_double['month'] = vw_double.index.to_series().dt.month
vw_double_ff = pd.merge(vw_double, ff5, on=['year', 'month'], how='inner')


VW 2x5: 100%|██████████| 37/37 [00:05<00:00,  6.98it/s]


In [ ]:
vw_results = []

for col in vw_double.columns:
    if col in ['year', 'month']: continue
    y = vw_double_ff[col] - vw_double_ff['RF']

    capm_X = sm.add_constant(vw_double_ff[['ExMkt']])
    ff3_X = sm.add_constant(vw_double_ff[['ExMkt', 'SMB', 'HML']])
    ff5_X = sm.add_constant(vw_double_ff[['ExMkt', 'SMB', 'HML', 'RMW', 'CMA']])

    capm = sm.OLS(y, capm_X).fit()
    ff3 = sm.OLS(y, ff3_X).fit()
    ff5_model = sm.OLS(y, ff5_X).fit()

    vw_results.append({
        'portfolio': col,
        'capm_alpha': capm.params['const'], 'capm_t': capm.tvalues['const'],
        'ff3_alpha': ff3.params['const'], 'ff3_t': ff3.tvalues['const'],
        'ff5_alpha': ff5_model.params['const'], 'ff5_t': ff5_model.tvalues['const'],
        'ff5_alpha_annual': ff5_model.params['const'] * 12,
        'R2_ff5': ff5_model.rsquared
    })

vw_double_table = pd.DataFrame(vw_results)
print("\nVW 2×5 Double Sort Results (CAPM / FF3 / FF5):")
print(vw_double_table.round(4))



VW 2×5 Double Sort Results (CAPM / FF3 / FF5):
  portfolio  capm_alpha  capm_t  ff3_alpha    ff3_t  ff5_alpha    ff5_t  \
0        S0      0.0237  6.1138     0.0292  12.5621     0.0318  15.4804   
1        S1      0.0195  5.9545     0.0236  12.0502     0.0258  14.2379   
2        S2      0.0174  7.5491     0.0194  15.1476     0.0200  15.6331   
3        S3      0.0171  8.1806     0.0183  15.3429     0.0185  15.8290   
4        S4      0.0247  9.5447     0.0251  15.6979     0.0247  15.3083   
5        B0      0.0054  5.4131     0.0070  10.4908     0.0070  10.3402   
6        B1      0.0039  6.2949     0.0039   6.4382     0.0032   5.4332   
7        B2      0.0050  7.0716     0.0043   6.6836     0.0036   5.6201   
8        B3      0.0043  5.5896     0.0030   5.4819     0.0026   4.6838   
9        B4      0.0054  4.6910     0.0031   5.0607     0.0030   4.7688   

   ff5_alpha_annual  R2_ff5  
0            0.3815  0.8326  
1            0.3093  0.8238  
2            0.2396  0.8474  
3     